In [1]:
# ============================================================
# UTS MINI-MODEL
#
# INPUT:
#   P, v, h, t
#
# OUTPUT:
#   UTS (MPa)
#
# Validation:
#   LOOCV
# ============================================================

import numpy as np
import pandas as pd
import warnings
import joblib

warnings.filterwarnings("ignore")

# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

raw_data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "UTS_MPa": [
        850, 915, 905, 910, 793,
        870, 875, 850, 860, 939,
        920, 910, 845, 925, 900,
        910, 920, 905, 895, 915
    ]
})


# ============================================================
# 2. AVERAGE REPEATED PROCESS CONDITIONS
# ============================================================

group_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    raw_data
    .groupby(group_cols, as_index=False)
    ["UTS_MPa"]
    .mean()
)

print("=" * 70)
print("UNIQUE PROCESS-CONDITION DATASET")
print("=" * 70)

print(data.to_string(index=False))

print("\nOriginal measurements :", len(raw_data))
print("Unique conditions     :", len(data))


# ============================================================
# 3. INPUT / TARGET
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

target = "UTS_MPa"

X = data[features]
y = data[target]


# ============================================================
# 4. IMPORT MODELS
# ============================================================

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 5. CANDIDATE MODELS
# ============================================================

models = {

    # --------------------------------------------------------
    # Linear
    # --------------------------------------------------------
    "Linear Regression":
        LinearRegression(),

    # --------------------------------------------------------
    # Ridge
    # --------------------------------------------------------
    "Ridge":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0))
        ]),

    # --------------------------------------------------------
    # Lasso
    # --------------------------------------------------------
    "Lasso":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=0.1))
        ]),

    # --------------------------------------------------------
    # Polynomial Degree 2
    # --------------------------------------------------------
    "Polynomial Degree 2":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=2,
                    include_bias=False
                )
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),

    # --------------------------------------------------------
    # Polynomial Degree 3
    # --------------------------------------------------------
    "Polynomial Degree 3 + Ridge":
        Pipeline([
            (
                "poly",
                PolynomialFeatures(
                    degree=3,
                    include_bias=False
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                Ridge(alpha=10.0)
            )
        ]),

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    # --------------------------------------------------------
    # Extra Trees
    # --------------------------------------------------------
    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------
    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_samples_leaf=2,
            random_state=42
        )
}


# ============================================================
# 6. OPTIONAL XGBOOST
# ============================================================

try:

    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=42
    )

    print("\nXGBoost available.")

except ImportError:

    print("\nXGBoost not installed.")


# ============================================================
# 7. OPTIONAL CATBOOST
# ============================================================

try:

    from catboost import CatBoostRegressor

    models["CatBoost"] = CatBoostRegressor(
        iterations=200,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=5,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )

    print("CatBoost available.")

except ImportError:

    print("CatBoost not installed.")


# ============================================================
# 8. LOOCV MODEL COMPARISON
# ============================================================

loo = LeaveOneOut()

results = []
all_predictions = {}

for name, model in models.items():

    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    rmse = np.sqrt(
        mean_squared_error(y, y_pred)
    )

    mae = mean_absolute_error(
        y,
        y_pred
    )

    r2 = r2_score(
        y,
        y_pred
    )

    # Training R²
    model.fit(X, y)

    y_train_pred = model.predict(X)

    train_r2 = r2_score(
        y,
        y_train_pred
    )

    results.append({

        "Model": name,

        "LOOCV_RMSE_MPa":
            rmse,

        "LOOCV_MAE_MPa":
            mae,

        "LOOCV_R2":
            r2,

        "Training_R2":
            train_r2
    })

    all_predictions[name] = y_pred


# ============================================================
# 9. RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values(
        by="LOOCV_RMSE_MPa"
    )
    .reset_index(drop=True)
)

print("\n")
print("=" * 70)
print("UTS MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 10. BEST MODEL
# ============================================================

best_name = results_df.iloc[0]["Model"]

best_model = models[best_name]

best_model.fit(X, y)

best_predictions = all_predictions[best_name]


print("\n")
print("=" * 70)
print("BEST UTS MODEL")
print("=" * 70)

print("Selected model:", best_name)

print(
    f"LOOCV RMSE = "
    f"{results_df.iloc[0]['LOOCV_RMSE_MPa']:.4f} MPa"
)

print(
    f"LOOCV MAE  = "
    f"{results_df.iloc[0]['LOOCV_MAE_MPa']:.4f} MPa"
)

print(
    f"LOOCV R²   = "
    f"{results_df.iloc[0]['LOOCV_R2']:.4f}"
)

print(
    f"Training R² = "
    f"{results_df.iloc[0]['Training_R2']:.4f}"
)


# ============================================================
# 11. ACTUAL VS PREDICTED
# ============================================================

comparison = data.copy()

comparison["Predicted_UTS_MPa"] = (
    best_predictions
)

comparison["Error_MPa"] = (
    comparison["Predicted_UTS_MPa"]
    - comparison["UTS_MPa"]
)

comparison["Absolute_Error_MPa"] = (
    comparison["Error_MPa"].abs()
)


print("\n")
print("=" * 70)
print("ACTUAL VS LOOCV PREDICTED")
print("=" * 70)

print(
    comparison[
        [
            "P_W",
            "v_mm_per_s",
            "h_mm",
            "t_mm",
            "UTS_MPa",
            "Predicted_UTS_MPa",
            "Error_MPa"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 12. PHYSICS-INSPIRED FEATURE TEST
# ============================================================

data["VED_J_per_mm3"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
        * data["t_mm"]
    )
)

data["ED_J_per_mm2"] = (
    data["P_W"]
    / (
        data["v_mm_per_s"]
        * data["h_mm"]
    )
)

data["log_VED"] = np.log(
    data["VED_J_per_mm3"]
)


feature_sets = {

    "P_v_h_t": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm"
    ],

    "VED_only": [
        "VED_J_per_mm3"
    ],

    "P_v_h_t_VED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "VED_J_per_mm3"
    ],

    "P_v_h_t_logVED": [
        "P_W",
        "v_mm_per_s",
        "h_mm",
        "t_mm",
        "log_VED"
    ],

    "VED_ED": [
        "VED_J_per_mm3",
        "ED_J_per_mm2"
    ]
}


feature_results = []

for feature_name, feature_list in feature_sets.items():

    X_feature = data[feature_list]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    y_pred = cross_val_predict(
        model,
        X_feature,
        y,
        cv=LeaveOneOut()
    )

    rmse = np.sqrt(
        mean_squared_error(y, y_pred)
    )

    mae = mean_absolute_error(
        y,
        y_pred
    )

    r2 = r2_score(
        y,
        y_pred
    )

    feature_results.append({

        "Feature_Set": feature_name,

        "LOOCV_RMSE_MPa":
            rmse,

        "LOOCV_MAE_MPa":
            mae,

        "LOOCV_R2":
            r2
    })


feature_results_df = (
    pd.DataFrame(feature_results)
    .sort_values("LOOCV_RMSE_MPa")
)


print("\n")
print("=" * 70)
print("PHYSICS-INSPIRED FEATURE COMPARISON")
print("=" * 70)

print(
    feature_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 13. FINAL FIT
# ============================================================

best_model.fit(
    data[features],
    y
)


# ============================================================
# 14. EXAMPLE PREDICTION
# ============================================================

example = pd.DataFrame({

    "P_W": [270],

    "v_mm_per_s": [1500],

    "h_mm": [0.08],

    "t_mm": [0.04]
})

predicted_uts = (
    best_model.predict(example)[0]
)

print("\n")
print("=" * 70)
print("EXAMPLE UTS PREDICTION")
print("=" * 70)

print(example)

print(
    f"\nPredicted UTS = "
    f"{predicted_uts:.2f} MPa"
)


# ============================================================
# 15. SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    "UTS_mini_model.pkl"
)

joblib.dump(
    features,
    "UTS_feature_order.pkl"
)

print("\n")
print("Saved:")
print("  UTS_mini_model.pkl")
print("  UTS_feature_order.pkl")

UNIQUE PROCESS-CONDITION DATASET
 P_W  v_mm_per_s  h_mm  t_mm    UTS_MPa
 250        1500 0.080  0.04 850.000000
 260        1450 0.075  0.04 845.000000
 260        1450 0.085  0.04 870.000000
 260        1550 0.075  0.04 850.000000
 260        1550 0.085  0.04 860.000000
 270        1400 0.080  0.04 939.000000
 270        1500 0.070  0.04 875.000000
 270        1500 0.080  0.04 909.166667
 270        1500 0.090  0.04 905.000000
 270        1600 0.080  0.04 793.000000
 280        1450 0.075  0.04 915.000000
 280        1450 0.085  0.04 910.000000
 280        1550 0.075  0.04 900.000000
 280        1550 0.085  0.04 920.000000
 290        1500 0.080  0.04 925.000000

Original measurements : 20
Unique conditions     : 15

XGBoost available.
CatBoost available.


UTS MODEL COMPARISON
                      Model  LOOCV_RMSE_MPa  LOOCV_MAE_MPa  LOOCV_R2  Training_R2
                      Ridge         29.2814        22.7686    0.4136       0.6818
Polynomial Degree 3 + Ridge         29.3423  

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]
y = data["UTS_MPa"]

UTS_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

UTS_model.fit(X, y)

joblib.dump(
    UTS_model,
    "UTS_mini_model.pkl"
)

joblib.dump(
    features,
    "UTS_feature_order.pkl"
)

print("UTS mini-model saved successfully.")

UTS mini-model saved successfully.


In [5]:
# ============================================================
# UTS MINI-MODEL COMPARISON
# P, v, h, t  →  UTS
#
# IMPORTANT:
# Only ACTUAL experimental UTS values are used.
# No predicted/dummy UTS values are used.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor
from catboost import CatBoostRegressor


# ============================================================
# 1. ACTUAL EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    # ACTUAL EXPERIMENTAL UTS VALUES
    "UTS_MPa": [
        939,
        908,
        929,
        908,
        896,
        890,
        905,
        903,
        910,
        910,
        923,
        793,
        895,
        890,
        870,
        895,
        897,
        885,
        905,
        850
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
#
# Repeated P-v-h-t conditions are averaged.
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(process_cols, as_index=False)["UTS_MPa"]
    .mean()
)

print("=" * 70)
print("DATA CHECK")
print("=" * 70)

print("Original experimental rows :", 20)
print("Unique process conditions  :", len(data))

print("\nUnique experimental dataset:")
print(
    data.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 3. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]
y = data["UTS_MPa"].values


# ============================================================
# 4. LOOCV
# ============================================================

loo = LeaveOneOut()


# ============================================================
# 5. MODELS
# ============================================================

models = {

    "Linear Regression":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

    "Ridge":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ]),

    "Lasso":
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=1))
        ]),

    "Polynomial Degree 2 + Ridge":
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=2,
                include_bias=False
            )),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ]),

    "Polynomial Degree 3 + Ridge":
        Pipeline([
            ("poly", PolynomialFeatures(
                degree=3,
                include_bias=False
            )),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10))
        ]),

    "Extra Trees":
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.03,
            loss="huber",
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            n_estimators=200,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42
        ),

    "CatBoost":
        CatBoostRegressor(
            iterations=200,
            depth=3,
            learning_rate=0.03,
            loss_function="RMSE",
            verbose=False,
            random_seed=42
        )
}


# ============================================================
# 6. MODEL COMPARISON
# ============================================================

results = []
predictions = {}

for name, model in models.items():

    # -------------------------------
    # LOOCV prediction
    # -------------------------------

    pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    predictions[name] = pred

    # -------------------------------
    # LOOCV metrics
    # -------------------------------

    rmse = np.sqrt(
        mean_squared_error(y, pred)
    )

    mae = mean_absolute_error(
        y, pred
    )

    r2 = r2_score(
        y, pred
    )

    # -------------------------------
    # Training R²
    # -------------------------------

    model.fit(X, y)

    train_pred = model.predict(X)

    train_r2 = r2_score(
        y,
        train_pred
    )

    results.append({

        "Model": name,

        "LOOCV_RMSE_MPa":
            rmse,

        "LOOCV_MAE_MPa":
            mae,

        "LOOCV_R2":
            r2,

        "Training_R2":
            train_r2
    })


# ============================================================
# 7. RESULTS
# ============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("LOOCV_RMSE_MPa")
    .reset_index(drop=True)
)

print("\n")
print("=" * 70)
print("UTS MODEL COMPARISON")
print("=" * 70)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 8. BEST MODEL
# ============================================================

best_name = results_df.iloc[0]["Model"]

best_model = models[best_name]

best_pred = predictions[best_name]


print("\n")
print("=" * 70)
print("BEST UTS MODEL")
print("=" * 70)

print("Selected model:", best_name)

print(
    f"LOOCV RMSE = "
    f"{results_df.iloc[0]['LOOCV_RMSE_MPa']:.4f} MPa"
)

print(
    f"LOOCV MAE  = "
    f"{results_df.iloc[0]['LOOCV_MAE_MPa']:.4f} MPa"
)

print(
    f"LOOCV R²   = "
    f"{results_df.iloc[0]['LOOCV_R2']:.4f}"
)

print(
    f"Training R² = "
    f"{results_df.iloc[0]['Training_R2']:.4f}"
)


# ============================================================
# 9. ACTUAL VS LOOCV PREDICTION
# ============================================================

comparison = data[
    process_cols + ["UTS_MPa"]
].copy()

comparison["Predicted_UTS_MPa"] = best_pred

comparison["Residual_MPa"] = (
    comparison["UTS_MPa"].values
    - best_pred
)

comparison["Absolute_Error_MPa"] = (
    np.abs(comparison["Residual_MPa"])
)


print("\n")
print("=" * 70)
print("ACTUAL VS LOOCV PREDICTED")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 10. RESIDUAL STATISTICS
# ============================================================

residuals = comparison["Residual_MPa"].values

print("\n")
print("=" * 70)
print("UTS RESIDUAL STATISTICS")
print("=" * 70)

print(
    f"Mean residual   = {np.mean(residuals):.4f} MPa"
)

print(
    f"Std residual    = {np.std(residuals, ddof=1):.4f} MPa"
)

print(
    f"Median residual = {np.median(residuals):.4f} MPa"
)

print(
    f"Min residual    = {np.min(residuals):.4f} MPa"
)

print(
    f"Max residual    = {np.max(residuals):.4f} MPa"
)

print(
    f"MAE             = "
    f"{mean_absolute_error(y, best_pred):.4f} MPa"
)

DATA CHECK
Original experimental rows : 20
Unique process conditions  : 15

Unique experimental dataset:
 P_W  v_mm_per_s  h_mm  t_mm  UTS_MPa
 250        1500 0.080 0.040  903.000
 260        1450 0.075 0.040  895.000
 260        1450 0.085 0.040  890.000
 260        1550 0.075 0.040  939.000
 260        1550 0.085 0.040  910.000
 270        1400 0.080 0.040  910.000
 270        1500 0.070 0.040  905.000
 270        1500 0.080 0.040  890.000
 270        1500 0.090 0.040  929.000
 270        1600 0.080 0.040  896.000
 280        1450 0.075 0.040  908.000
 280        1450 0.085 0.040  793.000
 280        1550 0.075 0.040  870.000
 280        1550 0.085 0.040  923.000
 290        1500 0.080 0.040  890.000


UTS MODEL COMPARISON
                      Model  LOOCV_RMSE_MPa  LOOCV_MAE_MPa  LOOCV_R2  Training_R2
                      Ridge         36.8039        25.5470   -0.2926       0.1545
              Random Forest         37.3858        24.6699   -0.3338       0.3708
                Ex

In [6]:
# ============================================================
# UTS — RESIDUAL / ERROR CALIBRATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso


# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "UTS_MPa": [
        939,
        908,
        929,
        908,
        896,
        890,
        905,
        903,
        910,
        910,
        923,
        793,
        895,
        890,
        870,
        895,
        897,
        885,
        905,
        850
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(
        process_cols,
        as_index=False
    )["UTS_MPa"]
    .mean()
)


# ============================================================
# 3. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]

y = data["UTS_MPa"].values


# ============================================================
# 4. LASSO MODEL
# ============================================================

uts_model = Pipeline([

    (
        "scaler",
        StandardScaler()
    ),

    (
        "lasso",
        Lasso(alpha=1)
    )
])


# ============================================================
# 5. LOOCV
# ============================================================

loo = LeaveOneOut()

predicted_uts = cross_val_predict(
    uts_model,
    X,
    y,
    cv=loo
)


# ============================================================
# 6. RESIDUALS
# ============================================================

residual = (
    y - predicted_uts
)

data["Predicted_UTS_MPa"] = predicted_uts

data["Residual_MPa"] = residual

data["AbsoluteError_MPa"] = np.abs(
    residual
)


# ============================================================
# 7. METRICS
# ============================================================

rmse = np.sqrt(
    np.mean(residual**2)
)

mae = np.mean(
    np.abs(residual)
)

print("\n")
print("=" * 70)
print("UTS LOOCV PERFORMANCE")
print("=" * 70)

print(f"RMSE = {rmse:.4f} MPa")
print(f"MAE  = {mae:.4f} MPa")


# ============================================================
# 8. RESIDUAL STATISTICS
# ============================================================

print("\n")
print("=" * 70)
print("UTS RESIDUAL STATISTICS")
print("=" * 70)

print(
    f"Mean residual       = "
    f"{residual.mean():.4f} MPa"
)

print(
    f"Residual std        = "
    f"{residual.std(ddof=1):.4f} MPa"
)

print(
    f"Residual median     = "
    f"{np.median(residual):.4f} MPa"
)

print(
    f"Minimum residual    = "
    f"{residual.min():.4f} MPa"
)

print(
    f"Maximum residual    = "
    f"{residual.max():.4f} MPa"
)

print(
    f"95% residual range  = "
    f"{np.percentile(residual, 2.5):.4f} "
    f"to "
    f"{np.percentile(residual, 97.5):.4f} MPa"
)


# ============================================================
# 9. ROBUST SIGMA
# ============================================================

median_residual = np.median(
    residual
)

mad = np.median(
    np.abs(
        residual -
        median_residual
    )
)

robust_sigma = (
    1.4826 * mad
)

print("\n")
print("=" * 70)
print("ROBUST ERROR")
print("=" * 70)

print(
    f"Standard deviation = "
    f"{residual.std(ddof=1):.4f} MPa"
)

print(
    f"Robust sigma       = "
    f"{robust_sigma:.4f} MPa"
)


# ============================================================
# 10. FULL RESIDUAL TABLE
# ============================================================

print("\n")
print("=" * 70)
print("LOOCV RESIDUAL TABLE")
print("=" * 70)

print(
    data[
        process_cols
        + [
            "UTS_MPa",
            "Predicted_UTS_MPa",
            "Residual_MPa"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 11. SAVE RESIDUAL INFORMATION
# ============================================================

residual_info = {

    "residual_pool_MPa":
        residual,

    "mean_residual_MPa":
        residual.mean(),

    "std_residual_MPa":
        residual.std(ddof=1),

    "median_residual_MPa":
        median_residual,

    "robust_sigma_MPa":
        robust_sigma,

    "features":
        features
}


print("\n")
print("=" * 70)
print("READY FOR UTS ERROR CALIBRATION")
print("=" * 70)

print(
    "We will use these residuals to determine "
    "the error model before saving the final UTS mini-model."
)



UTS LOOCV PERFORMANCE
RMSE = 39.6190 MPa
MAE  = 29.3280 MPa


UTS RESIDUAL STATISTICS
Mean residual       = 0.7075 MPa
Residual std        = 41.0030 MPa
Residual median     = -1.6154 MPa
Minimum residual    = -111.6360 MPa
Maximum residual    = 53.6574 MPa
95% residual range  = -84.9780 to 49.3019 MPa


ROBUST ERROR
Standard deviation = 41.0030 MPa
Robust sigma       = 30.4376 MPa


LOOCV RESIDUAL TABLE
 P_W  v_mm_per_s  h_mm  t_mm  UTS_MPa  Predicted_UTS_MPa  Residual_MPa
 250        1500 0.080 0.040  903.000            921.853       -18.853
 260        1450 0.075 0.040  895.000            903.319        -8.319
 260        1450 0.085 0.040  890.000            899.401        -9.401
 260        1550 0.075 0.040  939.000            907.354        31.646
 260        1550 0.085 0.040  910.000            911.615        -1.615
 270        1400 0.080 0.040  910.000            869.513        40.487
 270        1500 0.070 0.040  905.000            899.340         5.660
 270        1500 0.080 

In [7]:
# ============================================================
# UTS OUTLIER / SENSITIVITY ANALYSIS
#
# Purpose:
# 1. Evaluate the official model using ALL experimental data
# 2. Temporarily remove the 793 MPa point for sensitivity only
# 3. Compare model performance
# 4. Examine residuals
# 5. Estimate robust UTS error
#
# IMPORTANT:
# The 793 MPa point is NOT deleted from the official dataset.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "UTS_MPa": [
        939,
        908,
        929,
        908,
        896,
        890,
        905,
        903,
        910,
        910,
        923,
        793,
        895,
        890,
        870,
        895,
        897,
        885,
        905,
        850
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(
        process_cols,
        as_index=False
    )["UTS_MPa"]
    .mean()
)

print("=" * 70)
print("UNIQUE EXPERIMENTAL DATA")
print("=" * 70)

print(data.to_string(index=False))

print("\nUnique conditions:", len(data))


# ============================================================
# 3. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]


# ============================================================
# 4. RIDGE MODEL
#
# This is the model that previously gave the best result
# ============================================================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])


# ============================================================
# 5. FUNCTION FOR LOOCV
# ============================================================

def evaluate_model(df):

    X = df[features]
    y = df["UTS_MPa"].values

    loo = LeaveOneOut()

    pred = cross_val_predict(
        model,
        X,
        y,
        cv=loo
    )

    rmse = np.sqrt(
        mean_squared_error(y, pred)
    )

    mae = mean_absolute_error(
        y,
        pred
    )

    r2 = r2_score(
        y,
        pred
    )

    residuals = y - pred

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Predictions": pred,
        "Residuals": residuals
    }


# ============================================================
# 6. CASE A — ALL DATA
# ============================================================

result_all = evaluate_model(data)


print("\n")
print("=" * 70)
print("CASE A — ALL 15 UNIQUE CONDITIONS")
print("=" * 70)

print(
    f"LOOCV RMSE = {result_all['RMSE']:.4f} MPa"
)

print(
    f"LOOCV MAE  = {result_all['MAE']:.4f} MPa"
)

print(
    f"LOOCV R²   = {result_all['R2']:.4f}"
)


# ============================================================
# 7. IDENTIFY 793 MPa POINT
# ============================================================

outlier_mask = data["UTS_MPa"] == 793

outlier_point = data[outlier_mask].copy()

print("\n")
print("=" * 70)
print("793 MPa POINT")
print("=" * 70)

print(
    outlier_point.to_string(index=False)
)


# ============================================================
# 8. CASE B — WITHOUT 793 MPa
#
# ONLY FOR SENSITIVITY ANALYSIS
# ============================================================

data_without_793 = data[
    data["UTS_MPa"] != 793
].copy()

result_without = evaluate_model(
    data_without_793
)


print("\n")
print("=" * 70)
print("CASE B — WITHOUT 793 MPa POINT")
print("=" * 70)

print(
    f"LOOCV RMSE = {result_without['RMSE']:.4f} MPa"
)

print(
    f"LOOCV MAE  = {result_without['MAE']:.4f} MPa"
)

print(
    f"LOOCV R²   = {result_without['R2']:.4f}"
)


# ============================================================
# 9. PERFORMANCE CHANGE
# ============================================================

rmse_change = (
    result_all["RMSE"]
    - result_without["RMSE"]
)

mae_change = (
    result_all["MAE"]
    - result_without["MAE"]
)

r2_change = (
    result_without["R2"]
    - result_all["R2"]
)


print("\n")
print("=" * 70)
print("SENSITIVITY TO 793 MPa")
print("=" * 70)

print(
    f"RMSE change = {rmse_change:.4f} MPa"
)

print(
    f"MAE change  = {mae_change:.4f} MPa"
)

print(
    f"R² change   = {r2_change:.4f}"
)


# ============================================================
# 10. RESIDUAL TABLE — ALL DATA
# ============================================================

comparison = data[
    process_cols + ["UTS_MPa"]
].copy()

comparison["Predicted_UTS_MPa"] = (
    result_all["Predictions"]
)

comparison["Residual_MPa"] = (
    comparison["UTS_MPa"].values
    - comparison["Predicted_UTS_MPa"].values
)

comparison["Absolute_Error_MPa"] = (
    np.abs(comparison["Residual_MPa"])
)


print("\n")
print("=" * 70)
print("ALL-DATA LOOCV RESIDUAL TABLE")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 11. RESIDUAL STATISTICS
# ============================================================

residuals = result_all["Residuals"]

std_error = np.std(
    residuals,
    ddof=1
)

median_residual = np.median(
    residuals
)

mad = np.median(
    np.abs(
        residuals - median_residual
    )
)

robust_sigma = 1.4826 * mad

mae = np.mean(
    np.abs(residuals)
)


print("\n")
print("=" * 70)
print("RESIDUAL STATISTICS — ALL DATA")
print("=" * 70)

print(
    f"Mean residual       = {np.mean(residuals):.4f} MPa"
)

print(
    f"Residual std        = {std_error:.4f} MPa"
)

print(
    f"Residual median     = {median_residual:.4f} MPa"
)

print(
    f"Minimum residual    = {np.min(residuals):.4f} MPa"
)

print(
    f"Maximum residual    = {np.max(residuals):.4f} MPa"
)

print(
    f"MAE                 = {mae:.4f} MPa"
)

print(
    f"Robust sigma        = {robust_sigma:.4f} MPa"
)


# ============================================================
# 12. 793 MPa RESIDUAL
# ============================================================

outlier_residual = comparison.loc[
    comparison["UTS_MPa"] == 793,
    "Residual_MPa"
].iloc[0]

outlier_prediction = comparison.loc[
    comparison["UTS_MPa"] == 793,
    "Predicted_UTS_MPa"
].iloc[0]


print("\n")
print("=" * 70)
print("793 MPa POINT RESIDUAL")
print("=" * 70)

print(
    f"Actual UTS     = 793.000 MPa"
)

print(
    f"LOOCV predicted = {outlier_prediction:.4f} MPa"
)

print(
    f"Residual        = {outlier_residual:.4f} MPa"
)


# ============================================================
# 13. RECOMMENDATION
# ============================================================

print("\n")
print("=" * 70)
print("INTERPRETATION")
print("=" * 70)

if result_without["RMSE"] < 0.8 * result_all["RMSE"]:

    print(
        "The 793 MPa point has a STRONG influence on model error."
    )

    print(
        "Do NOT automatically delete it."
    )

    print(
        "Treat it as an experimental observation requiring"
        " separate physical/experimental verification."
    )

else:

    print(
        "The 793 MPa point does not dominate model performance."
    )

print("\n")

print(
    "Recommended robust error scale for further investigation:"
)

print(
    f"{robust_sigma:.2f} MPa"
)

print(
    "\nIMPORTANT: Do not inject this error into the final"
    " 500-point dataset until the sensitivity result is reviewed."
)

UNIQUE EXPERIMENTAL DATA
 P_W  v_mm_per_s  h_mm  t_mm  UTS_MPa
 250        1500 0.080  0.04    903.0
 260        1450 0.075  0.04    895.0
 260        1450 0.085  0.04    890.0
 260        1550 0.075  0.04    939.0
 260        1550 0.085  0.04    910.0
 270        1400 0.080  0.04    910.0
 270        1500 0.070  0.04    905.0
 270        1500 0.080  0.04    890.0
 270        1500 0.090  0.04    929.0
 270        1600 0.080  0.04    896.0
 280        1450 0.075  0.04    908.0
 280        1450 0.085  0.04    793.0
 280        1550 0.075  0.04    870.0
 280        1550 0.085  0.04    923.0
 290        1500 0.080  0.04    890.0

Unique conditions: 15


CASE A — ALL 15 UNIQUE CONDITIONS
LOOCV RMSE = 36.8039 MPa
LOOCV MAE  = 25.5470 MPa
LOOCV R²   = -0.2926


793 MPa POINT
 P_W  v_mm_per_s  h_mm  t_mm  UTS_MPa
 280        1450 0.085  0.04    793.0


CASE B — WITHOUT 793 MPa POINT
LOOCV RMSE = 20.3924 MPa
LOOCV MAE  = 16.9766 MPa
LOOCV R²   = -0.3896


SENSITIVITY TO 793 MPa
RMSE change = 16

In [8]:
# ============================================================
# FINAL UTS MINI-MODEL + ERROR CALIBRATION
#
# P, v, h, t  →  UTS
#
# Official experimental dataset
# 20 measurements → 15 unique process conditions
#
# Selected model: Ridge
# ============================================================

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. ACTUAL EXPERIMENTAL DATA
# ============================================================

data = pd.DataFrame({

    "P_W": [
        260, 280, 270, 270, 270,
        260, 270, 250, 260, 270,
        280, 280, 260, 290, 280,
        270, 270, 270, 270, 270
    ],

    "v_mm_per_s": [
        1550, 1450, 1500, 1500, 1600,
        1450, 1500, 1500, 1550, 1400,
        1550, 1450, 1450, 1500, 1550,
        1500, 1500, 1500, 1500, 1500
    ],

    "h_mm": [
        0.075, 0.075, 0.09, 0.08, 0.08,
        0.085, 0.07, 0.08, 0.085, 0.08,
        0.085, 0.085, 0.075, 0.08, 0.075,
        0.08, 0.08, 0.08, 0.08, 0.08
    ],

    "t_mm": [
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04,
        0.04, 0.04, 0.04, 0.04, 0.04
    ],

    "UTS_MPa": [
        939,
        908,
        929,
        908,
        896,
        890,
        905,
        903,
        910,
        910,
        923,
        793,
        895,
        890,
        870,
        895,
        897,
        885,
        905,
        850
    ]
})


# ============================================================
# 2. REMOVE DUPLICATE PROCESS CONDITIONS
# ============================================================

process_cols = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

data = (
    data
    .groupby(
        process_cols,
        as_index=False
    )["UTS_MPa"]
    .mean()
)

print("=" * 70)
print("FINAL UNIQUE UTS DATASET")
print("=" * 70)

print(data.to_string(index=False))

print("\nOriginal measurements :", 20)
print("Unique conditions     :", len(data))


# ============================================================
# 3. FEATURES
# ============================================================

features = [
    "P_W",
    "v_mm_per_s",
    "h_mm",
    "t_mm"
]

X = data[features]
y = data["UTS_MPa"].values


# ============================================================
# 4. FINAL RIDGE MODEL
# ============================================================

uts_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])


# ============================================================
# 5. LOOCV
# ============================================================

loo = LeaveOneOut()

loocv_pred = cross_val_predict(
    uts_model,
    X,
    y,
    cv=loo
)


# ============================================================
# 6. LOOCV METRICS
# ============================================================

rmse = np.sqrt(
    mean_squared_error(
        y,
        loocv_pred
    )
)

mae = mean_absolute_error(
    y,
    loocv_pred
)

r2 = r2_score(
    y,
    loocv_pred
)

residuals = y - loocv_pred


print("\n")
print("=" * 70)
print("FINAL UTS LOOCV PERFORMANCE")
print("=" * 70)

print(f"RMSE = {rmse:.4f} MPa")
print(f"MAE  = {mae:.4f} MPa")
print(f"R²   = {r2:.4f}")


# ============================================================
# 7. RESIDUAL STATISTICS
# ============================================================

residual_std = np.std(
    residuals,
    ddof=1
)

residual_median = np.median(
    residuals
)

mad = np.median(
    np.abs(
        residuals - residual_median
    )
)

robust_sigma = 1.4826 * mad


print("\n")
print("=" * 70)
print("UTS RESIDUAL STATISTICS")
print("=" * 70)

print(
    f"Mean residual   = {np.mean(residuals):.4f} MPa"
)

print(
    f"Std residual    = {residual_std:.4f} MPa"
)

print(
    f"Median residual = {residual_median:.4f} MPa"
)

print(
    f"Minimum         = {np.min(residuals):.4f} MPa"
)

print(
    f"Maximum         = {np.max(residuals):.4f} MPa"
)

print(
    f"MAE             = {mae:.4f} MPa"
)

print(
    f"Robust sigma     = {robust_sigma:.4f} MPa"
)


# ============================================================
# 8. LOOCV RESIDUAL TABLE
# ============================================================

comparison = data[
    process_cols + ["UTS_MPa"]
].copy()

comparison["Predicted_UTS_MPa"] = loocv_pred

comparison["Residual_MPa"] = (
    comparison["UTS_MPa"].values
    - comparison["Predicted_UTS_MPa"].values
)

comparison["Absolute_Error_MPa"] = (
    np.abs(
        comparison["Residual_MPa"]
    )
)


print("\n")
print("=" * 70)
print("LOOCV RESIDUAL TABLE")
print("=" * 70)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


# ============================================================
# 9. PI1 CALCULATION
#
# Used only to check whether residuals have systematic
# dependence on process-energy regime.
# ============================================================

Cp = 436.0
k_mm = 0.0103

comparison["Pi1"] = (
    Cp * comparison["P_W"]
    /
    (
        k_mm
        * comparison["v_mm_per_s"]**2
        * comparison["h_mm"]
    )
)


pi1_residual_corr = np.corrcoef(
    comparison["Pi1"],
    comparison["Residual_MPa"]
)[0, 1]


print("\n")
print("=" * 70)
print("RESIDUAL DEPENDENCE ON Pi1")
print("=" * 70)

print(
    f"Correlation(residual, Pi1) = "
    f"{pi1_residual_corr:.4f}"
)


# ============================================================
# 10. ERROR CALIBRATION
# ============================================================

# Robust sigma is preferred because the 793 MPa observation
# produces a large residual and would inflate the ordinary
# standard deviation.

UTS_error_sigma = robust_sigma


print("\n")
print("=" * 70)
print("UTS ERROR CALIBRATION")
print("=" * 70)

print(
    f"Standard deviation = "
    f"{residual_std:.4f} MPa"
)

print(
    f"Robust sigma       = "
    f"{robust_sigma:.4f} MPa"
)

print(
    f"Selected error sigma = "
    f"{UTS_error_sigma:.4f} MPa"
)


# ============================================================
# 11. FIT FINAL MODEL ON ALL 15 CONDITIONS
# ============================================================

uts_model.fit(
    X,
    y
)


# ============================================================
# 12. SAVE MODEL
# ============================================================

joblib.dump(
    uts_model,
    "UTS_mini_model.pkl"
)

joblib.dump(
    features,
    "UTS_feature_order.pkl"
)


# Save calibration information
uts_calibration = {
    "model": "Ridge",
    "ridge_alpha": 10,
    "features": features,
    "n_original_measurements": 20,
    "n_unique_conditions": len(data),

    "LOOCV_RMSE_MPa": rmse,
    "LOOCV_MAE_MPa": mae,
    "LOOCV_R2": r2,

    "residual_std_MPa": residual_std,
    "robust_sigma_MPa": robust_sigma,
    "selected_error_sigma_MPa": UTS_error_sigma,

    "Pi1_residual_correlation": pi1_residual_corr,

    "experimental_UTS_min_MPa": float(y.min()),
    "experimental_UTS_max_MPa": float(y.max())
}

joblib.dump(
    uts_calibration,
    "UTS_calibration.pkl"
)


# ============================================================
# 13. TEST THE SAVED MODEL
# ============================================================

test_X = pd.DataFrame([{
    "P_W": 270,
    "v_mm_per_s": 1500,
    "h_mm": 0.080,
    "t_mm": 0.040
}])

test_prediction = uts_model.predict(
    test_X
)[0]


print("\n")
print("=" * 70)
print("EXAMPLE UTS PREDICTION")
print("=" * 70)

print(test_X.to_string(index=False))

print(
    f"\nPredicted UTS = "
    f"{test_prediction:.2f} MPa"
)


# ============================================================
# 14. FINAL SAVE CONFIRMATION
# ============================================================

print("\n")
print("=" * 70)
print("FILES SAVED")
print("=" * 70)

print("UTS_mini_model.pkl")
print("UTS_feature_order.pkl")
print("UTS_calibration.pkl")

FINAL UNIQUE UTS DATASET
 P_W  v_mm_per_s  h_mm  t_mm  UTS_MPa
 250        1500 0.080  0.04    903.0
 260        1450 0.075  0.04    895.0
 260        1450 0.085  0.04    890.0
 260        1550 0.075  0.04    939.0
 260        1550 0.085  0.04    910.0
 270        1400 0.080  0.04    910.0
 270        1500 0.070  0.04    905.0
 270        1500 0.080  0.04    890.0
 270        1500 0.090  0.04    929.0
 270        1600 0.080  0.04    896.0
 280        1450 0.075  0.04    908.0
 280        1450 0.085  0.04    793.0
 280        1550 0.075  0.04    870.0
 280        1550 0.085  0.04    923.0
 290        1500 0.080  0.04    890.0

Original measurements : 20
Unique conditions     : 15


FINAL UTS LOOCV PERFORMANCE
RMSE = 36.8039 MPa
MAE  = 25.5470 MPa
R²   = -0.2926


UTS RESIDUAL STATISTICS
Mean residual   = 0.3527 MPa
Std residual    = 38.0939 MPa
Median residual = 4.9225 MPa
Minimum         = -110.7512 MPa
Maximum         = 47.0714 MPa
MAE             = 25.5470 MPa
Robust sigma     = 28.9